# Demo 1 — Shared Configuration

This notebook defines the shared configuration used by the crypto demo.

It stores:
- Unity Catalog names
- Azure Storage and external Volume paths
- Project folder paths
- Bronze, Silver, and Gold table names
- Event Hub and secret-scope settings
- Historical symbols and source-system constants

> This notebook only defines values. It does not create the schema, Volume, folders, or tables. Those objects will be created in `setup/01_setup.py`.

## 1. Create widgets

Widgets make the notebook reusable. Values can be changed from the Databricks notebook interface without editing the source code.

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("catalog", "dbr_dev", "01 Catalog")
dbutils.widgets.text("schema", "parvinbadalov", "02 Schema")
dbutils.widgets.text("volume_name", "demo1_crypto", "03 Volume name")

dbutils.widgets.text(
    "storage_account",
    "dlspl21databricks",
    "04 Storage account"
)

dbutils.widgets.text(
    "container",
    "parvinbadalov",
    "05 Container"
)

dbutils.widgets.text(
    "external_volume_dir",
    "demos/demo1_crypto",
    "06 External Volume directory"
)

dbutils.widgets.text(
    "external_location_name",
    "parvinbadalov_external_location",
    "07 External location"
)

dbutils.widgets.text(
    "eventhub_name",
    "parvinbadalov_evh",
    "08 Event Hub name"
)

dbutils.widgets.text(
    "eventhub_consumer_group",
    "parvinbadalov",
    "09 Event Hub consumer group"
)

dbutils.widgets.text(
    "eventhub_secret_scope",
    "default2",
    "10 Secret scope"
)

dbutils.widgets.text(
    "eventhub_secret_key",
    "parvinbadalov-eventhub-cs",
    "11 Event Hub secret key"
)

## 2. Read widget values

Each widget value is read once and stored in a Python variable. `.strip()` removes accidental spaces.

In [0]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()

storage_account = dbutils.widgets.get("storage_account").strip()
container = dbutils.widgets.get("container").strip()

external_volume_dir = dbutils.widgets.get(
    "external_volume_dir"
).strip()

external_location_name = dbutils.widgets.get(
    "external_location_name"
).strip()

eventhub_name = dbutils.widgets.get(
    "eventhub_name"
).strip()

eventhub_consumer_group = dbutils.widgets.get(
    "eventhub_consumer_group"
).strip()

eventhub_secret_scope = dbutils.widgets.get(
    "eventhub_secret_scope"
).strip()

eventhub_secret_key = dbutils.widgets.get(
    "eventhub_secret_key"
).strip()

## 3. Validate required values

The notebook stops early when an important configuration value is empty.

In [0]:
required_values = {
    "catalog": catalog,
    "schema": schema,
    "volume_name": volume_name,
    "storage_account": storage_account,
    "container": container,
    "external_volume_dir": external_volume_dir,
    "external_location_name": external_location_name,
    "eventhub_name": eventhub_name,
    "eventhub_consumer_group": eventhub_consumer_group,
    "eventhub_secret_scope": eventhub_secret_scope,
    "eventhub_secret_key": eventhub_secret_key,
}

missing_values = [
    name
    for name, value in required_values.items()
    if not value
]

if missing_values:
    raise ValueError(
        f"Missing required configuration values: {missing_values}"
    )

## 4. Build Azure and Volume paths

The physical Azure location is:

`abfss://parvinbadalov@dlspl21databricks.dfs.core.windows.net/demos/demo1_crypto`

The Unity Catalog Volume path is:

`/Volumes/dbr_dev/parvinbadalov/demo1_crypto`

In [0]:
external_volume_location = (
    f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
    f"{external_volume_dir}"
)

volume_root = (
    f"/Volumes/{catalog}/{schema}/{volume_name}"
)

## 5. Define project folder paths

Planned structure:

```text
demo1_crypto/
├── raw/
│   ├── historical/
│   └── streaming_test/
├── landing/
│   └── historical/
├── system/
│   ├── schema/
│   │   └── crypto_ticks/
│   └── checkpoints/
│       └── crypto_ticks/
└── archive/
```

In [0]:
raw_path = f"{volume_root}/raw"
historical_raw_path = f"{raw_path}/historical"
streaming_test_path = f"{raw_path}/streaming_test"

landing_path = f"{volume_root}/landing"
historical_landing_path = f"{landing_path}/historical"

system_path = f"{volume_root}/system"

schema_root_path = f"{system_path}/schema"
crypto_ticks_schema_path = (
    f"{schema_root_path}/crypto_ticks"
)

checkpoint_root_path = f"{system_path}/checkpoints"
crypto_ticks_checkpoint_path = (
    f"{checkpoint_root_path}/crypto_ticks"
)

archive_path = f"{volume_root}/archive"

## 6. Define table names

- Bronze: raw ingested data
- Silver: cleaned and deduplicated data
- Gold: dashboard-ready results

In [0]:
historical_bronze_table = (
    f"{catalog}.{schema}.demo1_crypto_ohlcv_bronze"
)

streaming_bronze_table = (
    f"{catalog}.{schema}.demo1_crypto_ticks_bronze"
)

historical_silver_table = (
    f"{catalog}.{schema}.demo1_crypto_ohlcv_silver"
)

streaming_silver_table = (
    f"{catalog}.{schema}.demo1_crypto_ticks_silver"
)

latest_prices_gold_table = (
    f"{catalog}.{schema}.demo1_crypto_latest_prices_gold"
)

market_summary_gold_table = (
    f"{catalog}.{schema}.demo1_crypto_market_summary_gold"
)

## 7. Define project constants

The historical batch source contains daily January 2026 data for Bitcoin, Ethereum, and Solana.

In [0]:
historical_symbols = [
    "BTCUSDT",
    "ETHUSDT",
    "SOLUSDT",
]

historical_interval = "1d"
historical_period = "2026-01"

source_system_historical = "binance_data_vision"
source_system_streaming = "binance_public_api"

rescued_data_column = "_rescued_data"

## 8. Display configuration summary

The summary prints names and paths only. It does not retrieve or display any secret values.

In [0]:
print("Demo 1 configuration loaded successfully.")
print()

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Volume: {catalog}.{schema}.{volume_name}")
print(f"External Volume location: {external_volume_location}")
print(f"Volume root: {volume_root}")
print()

print(f"Historical raw path: {historical_raw_path}")
print(f"Historical landing path: {historical_landing_path}")
print(f"Streaming schema path: {crypto_ticks_schema_path}")
print(f"Streaming checkpoint path: {crypto_ticks_checkpoint_path}")
print()

print(f"Historical Bronze table: {historical_bronze_table}")
print(f"Streaming Bronze table: {streaming_bronze_table}")
print()

print(f"Historical symbols: {historical_symbols}")
print(f"Historical interval: {historical_interval}")
print(f"Historical period: {historical_period}")
print()

print(f"Event Hub: {eventhub_name}")
print(f"Consumer group: {eventhub_consumer_group}")
print(f"Secret scope: {eventhub_secret_scope}")
print(f"Secret key name: {eventhub_secret_key}")

## Next notebook

`Demos/Demo1/setup/01_setup.py`

That notebook will:
- create or confirm the schema
- create the external Volume
- create the planned folders
- validate the Volume